# Slide Exercise 01: Expanded TF-IDF Movie Recommender

This is the refined version of `TFIDF_MovieRecommender_Expanded.ipynb`.

Learning objectives:
- Build a stronger TF-IDF content representation from title, genres, director, description, and keywords.
- Generate Top-N similar movies.
- Explain recommendations with shared weighted terms.

Main functions used:
- `TfidfVectorizer(...)`: converts text into weighted term vectors.
- `fit_transform(...)`: learns the vocabulary and creates the movie-term matrix.
- `cosine_similarity(...)`: compares movie vectors by angle.
- `argsort()`: sorts similarity scores to create a ranking.


Load the shared Chapter 2 dataset. We keep the dataset small so students can inspect every intermediate result.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("../data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("chapter_02_content_based/data")

movies = pd.read_csv(DATA_DIR / "movies_chapter2.csv")
movies.head()


Create one clean text field per movie. In real systems this step is often called metadata fusion.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

movies["content"] = (
    movies["title"] + " " +
    movies["genres"].str.replace("|", " ", regex=False) + " " +
    movies["director"] + " " +
    movies["description"] + " " +
    movies["keywords"]
).apply(clean_text)

movies[["title", "content"]].head()


Fit TF-IDF and compute a movie-by-movie similarity matrix.


In [ ]:
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
tfidf_matrix = vectorizer.fit_transform(movies["content"])
similarity = cosine_similarity(tfidf_matrix)

print("TF-IDF matrix shape:", tfidf_matrix.shape)
pd.DataFrame(similarity, index=movies["title"], columns=movies["title"]).round(2)


Define small reusable functions. The recommendation function ranks movies; the explanation function shows shared TF-IDF features.


In [ ]:
def shared_weighted_terms(seed_idx, other_idx, top_n=6):
    terms = np.array(vectorizer.get_feature_names_out())
    seed_weights = tfidf_matrix[seed_idx].toarray().ravel()
    other_weights = tfidf_matrix[other_idx].toarray().ravel()
    shared_weight = np.minimum(seed_weights, other_weights)
    best = shared_weight.argsort()[::-1][:top_n]
    return ", ".join(terms[i] for i in best if shared_weight[i] > 0)

def recommend_similar_movies(title, n=5):
    seed_idx = movies.index[movies["title"].eq(title)][0]
    ranked = similarity[seed_idx].argsort()[::-1]
    rows = []
    for other_idx in ranked:
        if other_idx == seed_idx:
            continue
        rows.append({
            "input_movie": title,
            "recommended_movie": movies.loc[other_idx, "title"],
            "similarity": round(float(similarity[seed_idx, other_idx]), 3),
            "shared_terms": shared_weighted_terms(seed_idx, other_idx),
        })
        if len(rows) == n:
            break
    return pd.DataFrame(rows)

recommend_similar_movies("Interstellar")


Interpretation:

Movies with shared terms such as `space`, `astronaut`, `sci-fi`, or a shared director move upward in the ranking. TF-IDF is still lexical, so it works best when related movies use overlapping vocabulary.

Student task:
1. Change the input movie to `Toy Story`.
2. Remove bigrams by setting `ngram_range=(1, 1)`. Did the ranking change?
